# Notebook 5 — SLM Benchmarking on **Real** Project-Management Data (4 models)

**Models:** Qwen3-8B · Llama-3.1-8B-Instruct · GLM-Z1-9B (local, 4-bit NF4) · **Gemini-3.5-Flash-Lite** (API)
**Tasks:** Task Design · Task Allocation · Sprint Planning · Resource Planning
**Dataset:** real Apache Jira data (issues.apache.org) — **50 samples per task type, 200 in total**
**Judge:** Inception `mercury-2` (same judge as the synthetic benchmark)

---

### Why a second benchmark on real data?

The synthetic benchmark (Notebook 3) scores each answer against a **reference written by another LLM**
(ROUGE-L / BERTScore against Mercury-2 / Gemma output). That is partly circular: a model that writes like the
generator scores well. Here every task is scored against **what actually happened** in the Jira project, where
such ground truth exists:

| Track | Real ground truth | Scored by |
|---|---|---|
| Task Design | the issue's real Jira priority (no reference text) | schema compliance · priority match · judge |
| Task Allocation | the real assignee and real logged hours | assignee exact match · hours MAPE · judge |
| Sprint Planning | the issues really shipped in the release (`fixVersion`), real scope size | task-set F1 · scope MAPE · judge |
| Resource Planning | the project's real velocity (issues per release) | velocity MAPE (low confidence) · schema · judge |

Because each track uses different metrics, **there is no single pooled composite** here — every track gets its own table.

### Pipeline

```
real_task_design.json · real_task_allocation.json · real_sprint_planning_v3.json · real_resource_planning_v3.json
   │  Cell 3   sample 50 per track (seed 42) → 200 samples, same for every model
   ├─► Cell 4   local SLMs: 4-bit load → batched generation (batch 12) → unload
   ├─► Cell 4b  Gemini: sequential API calls (11 req/min)
   ├─► Cell 5   track-specific real-ground-truth metrics
   ├─► Cell 6   LLM judge (3 keys in parallel)
   ├─► Cell 7–8 orchestrate + checkpoint → real_benchmark_results_real_001.csv (800 rows)
   ├─► Cell 9   extract_json fix + re-scoring · per-track tables
   ├─► Cell 10  Wilcoxon (per track) + McNemar (assignee match)
   └─► Cell 11–12 paper tables (CSV + LaTeX) · charts
```

### Results at a glance (from the outputs below)

| Track | Best model | Key numbers |
|---|---|---|
| Task Design | Gemini ≈ Llama (judge 2.82 vs 2.69, difference **not** significant) | Qwen has near-perfect schema (0.98) but lower judge (2.13) |
| Task Allocation | Gemini (24 % exact assignee; every local SLM ≈ 0 %) | see ⚠️ prompt-truncation note in Cell 9 |
| Sprint Planning | Llama ≈ Qwen (task-set F1 0.28 / 0.27) | Gemini only 0.06 — its JSON is often invalid here |
| Resource Planning | all weak (judge 1.25–1.73) | velocity MAPE is not informative (see Cell 9) |

### ⚠️ The three things to know before using these numbers

1. **Prompt truncation for local models.** Generation truncates every prompt to **1,024 tokens** from the right.
   Task-allocation prompts (median ≈ 4,600 characters, team roster in JSON) are longer than that, so the chat
   template's *assistant* turn is cut off and the local models simply **continue the roster text**.
   Gemini (API) is not truncated. Task-allocation results for the local models reflect this bug, not model ability.
2. **Judge sees only 500 characters** of each answer (`_sanitize(..., 500)`), so long answers are judged on their beginning.
3. **The `extract_json` fix in Cell 9 was written to a new CSV, but the tables after it re-load the old CSV**, so the
   printed tables use the *original* scoring. Notebook 7 recomputes everything with the fixed scorer.

### Relationship to the other notebooks

| Notebook | Role |
|---|---|
| 3 · synthetic benchmark (4 models) | same models on the synthetic data |
| 4 · fine-tune + synthetic eval | creates the `Qwen3-8B-PM-LoRA` adapter |
| **5 · this notebook** | base models on real Apache Jira data |
| 6 · FT model on real data | adds `Qwen3-8B-FT` as a 5th model to this notebook's results |
| 7 · final comparison | all models, synthetic + real, recomputed from the saved result files |

> **How to read this notebook.** Code and outputs are unchanged from the original Modal/Colab run (keys replaced with
> placeholders). **📊 Reading the output** cells interpret the numbers; **⚠️ Reviewer note** cells flag issues.

## Cell 1 — Install Dependencies

`transformers`, `accelerate`, `torch`, `bitsandbytes`, `flash-attn` for local 4-bit inference, `rouge_score` /
`bert_score` (kept for diagnostics only — not used in any real-data score) and **`google-genai`**, the new Gemini SDK.

The red `metadata-generation-failed` message in the output comes from building **`flash-attn`** from source; the
container already had a working build, which is why `attn_implementation="flash_attention_2"` still works later.

In [ ]:
# Local-model stack (flash-attn may fail to build here; a pre-built copy was already installed).
!pip install transformers accelerate torch \
    bitsandbytes flash-attn --quiet
!pip install rouge_score bert_score --quiet
# New Gemini SDK (`from google import genai`).
!pip install google-genai --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 44.9 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00


## Cell 2 — Configuration

| Setting | Value | Meaning |
|---|---|---|
| `MODELS` | Qwen3-8B, GLM-Z1-9B, Llama-3.1-8B | local models, loaded one at a time in 4-bit |
| `API_MODELS` | Gemini-3.5-Flash-Lite | called over HTTP, no GPU needed |
| `REAL_DATASET_PATHS` | 4 JSON files | one real dataset per track |
| `SAMPLES_PER_TYPE` / `RANDOM_SEED` | 50 / 42 | same 200 samples for every model on every re-run |
| `GEMINI_REQUESTS_PER_MIN` | 11 | ≈ 5.45 s pause between Gemini calls |
| `MAX_NEW_TOKENS`, `TEMPERATURE`, `BATCH_SIZE` | 600, 0.3, 12 | identical to the synthetic benchmark |
| `CHECKPOINT_DIR` | `checkpoints_real` | one JSON per model; a re-run skips finished models |

In [ ]:
import os

# Placeholders — use your own keys (Colab Secrets / environment variables).
# ── API Keys ─────────────────────────────────────────────────────────────────
os.environ["INCEPTION_API_KEY"] = "YOUR_INCEPTION_API_KEY_1"
os.environ["HF_TOKEN"]          = "YOUR_HF_TOKEN"
os.environ["GEMINI_API_KEY"]    = "YOUR_GEMINI_API_KEY"

# ── Local model registry (unchanged from v5) ─────────────────────────────────
# Local open-weight models (4-bit, run on the GPU one after another).
MODELS = {
    "Qwen3-8B":      "Qwen/Qwen3-8B",
    "GLM-Z1-9B":     "zai-org/GLM-Z1-9B-0414",
    "Llama-3.1-8B":  "meta-llama/Llama-3.1-8B-Instruct",
}

# ── API model registry (new — no local load/unload, called over HTTP) ────────
# Closed model reached through its API.
API_MODELS = {
    "Gemini-3.5-Flash-Lite": "gemini-3.5-flash-lite",
}

# ── REAL dataset paths ────────────────────────────────────────────────────────
# One real Apache-Jira dataset per track.
REAL_DATASET_PATHS = {
    "task_design":       "real_task_design.json",
    "task_allocation":   "real_task_allocation.json",
    "sprint_planning":   "real_sprint_planning_v3.json",  # a 'sprint' = one Jira fixVersion (release)
    "resource_planning": "real_resource_planning_v3.json",
}
SAMPLES_PER_TYPE = 50   # 50 × 4 types = 200 total, sampled from each real file
RANDOM_SEED      = 42   # fixed seed → same 200 samples every re-run

# ── Gemini API rate limit ─────────────────────────────────────────────────────
GEMINI_REQUESTS_PER_MIN = 11   # hard cap — 60/11 ≈ 5.45s between calls
GEMINI_SLEEP_BETWEEN    = 60.0 / GEMINI_REQUESTS_PER_MIN  # ≈ 5.45 s between calls

# ── Inference settings (unchanged) ────────────────────────────────────────────
MAX_NEW_TOKENS  = 600  # same budget as the synthetic run
TEMPERATURE     = 0.3
DO_SAMPLE       = True
BATCH_SIZE      = 12
USE_4BIT        = True  # NF4 quantization for the local models

# ── Judge ─────────────────────────────────────────────────────────────────────
JUDGE_MODEL   = "mercury-2"
INCEPTION_URL = "https://api.inceptionlabs.ai/v1/chat/completions"

# ── Output ────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "checkpoints_real"  # per-model checkpoints → resume after a crash
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("✅ Config loaded")
print(f"Local models : {list(MODELS.keys())}")
print(f"API models   : {list(API_MODELS.keys())}")
print(f"Samples      : {SAMPLES_PER_TYPE * 4} total ({SAMPLES_PER_TYPE} per task type)")

✅ Config loaded
Local models : ['Qwen3-8B', 'GLM-Z1-9B', 'Llama-3.1-8B']
API models   : ['Gemini-3.5-Flash-Lite']
Samples      : 200 total (50 per task type)


## Cell 3 — Real Dataset Loader

**Objective:** load the four real-data files, tag each sample with its `task_type`, and draw **50 samples per
track with a fixed seed**, so every model sees exactly the same 200 prompts.

Each real sample carries its own ground-truth fields (`known_priority`, `ground_truth_assignee`,
`ground_truth_hours`, …). They are passed through untouched so the scoring cell can use them.

`REQUIRED_FIELDS` differs from the synthetic benchmark:
* sprint planning asks for `scope_size` instead of `total_points` — story points are not filled in on the Apache
  projects that were mined;
* task allocation asks for `role` and `rationale` instead of `task` and `reason`;
* resource planning requires three keys instead of one.

In [ ]:
import json, random
import pandas as pd

random.seed(RANDOM_SEED)

VALID_TASK_TYPES = {
    "task_design",
    "task_allocation",
    "sprint_planning",
    "resource_planning",
}

# Real-track required fields differ slightly from the synthetic schema
# (e.g. sprint_planning uses scope_size, not total_points — story points
# weren't populated on any ASF project pulled, see Methodology note).
# Keys each track's JSON answer must contain → schema compliance = found / required.
REQUIRED_FIELDS = {
    "task_design":       ["title", "description", "acceptance_criteria", "story_points", "priority"],
    "task_allocation":   ["assigned_to", "role", "rationale", "estimated_hours"],
    "sprint_planning":   ["sprint_goal", "selected_tasks", "scope_size", "excluded_tasks"],
    "resource_planning": ["capacity_constraints", "risk_flags", "mitigation_recommendations"],
}


# Load each track file and sample a fixed subset from it.
def load_real_dataset(paths: dict, samples_per_type: int, seed: int) -> list:
    dataset = []
    print("📊 Real dataset breakdown:")
    for task_type, path in paths.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Real dataset not found: {path}")
        with open(path, "r", encoding="utf-8") as f:
            samples = json.load(f)

        for s in samples:
            s["task_type"] = task_type  # label each sample with its track
            if "reference_output" not in s:
                s["reference_output"] = ""  # task design has no reference text → empty string

        rng = random.Random(seed)  # new RNG per track with the same seed → reproducible sample
        selected = samples if len(samples) <= samples_per_type else rng.sample(samples, samples_per_type)  # take all if ≤ 50, else 50 without replacement
        dataset.extend(selected)

        status = "✅" if len(samples) >= samples_per_type else "⚠️ "
        print(f"   {status} {task_type}: {len(selected)} used / {len(samples)} available")

    print(f"\n✅ Total REAL samples loaded: {len(dataset)}")
    return dataset


REAL_BENCHMARK_DATASET = load_real_dataset(REAL_DATASET_PATHS, SAMPLES_PER_TYPE, RANDOM_SEED)

📊 Real dataset breakdown:
   ✅ task_design: 50 used / 360 available
   ✅ task_allocation: 50 used / 52 available
   ✅ sprint_planning: 50 used / 71 available
   ✅ resource_planning: 50 used / 71 available

✅ Total REAL samples loaded: 200


### 📊 Reading the output

Task design draws 50 from 360 real issues; task allocation 50 from **52** (almost the whole file), sprint and
resource planning 50 from 71 each. Total: **200 samples**.

## Cell 4 — Local Models: 4-bit Loading & Batched Inference

Same engine as the synthetic benchmark: NF4 4-bit weights, left padding, batch 12, `repetition_penalty=1.05`,
one shared system prompt asking for JSON only. Qwen3's reasoning mode is switched off by inserting `/no_think`.

**Each result keeps a `_meta` dict** with every extra field of the sample (the real ground truth), which is what
Cell 5 scores against.

### ⚠️ Reviewer note — `max_length=1024` truncates the prompt from the right

`tokenizer(..., truncation=True, max_length=1024)` cuts the **end** of any prompt longer than 1,024 tokens. The end
of a chat prompt is the instruction plus the `assistant` header, so for long prompts the model never sees the
request to answer — it continues the input instead. Measured on the saved outputs:

| Track | Median prompt length | Answers whose first 15 characters are copied from the prompt |
|---|---|---|
| task allocation | ~4,600 chars | Qwen 56 % · Llama 66 % · GLM 78 % · **Gemini 0 %** |
| sprint planning | ~2,900 chars | 0–6 % |
| task design / resource planning | ~1,100 chars | 0 % |

(Measured in Notebook 7. Only 3 of Qwen's and 4 of Llama's 50 task-allocation answers even begin with `{` or `<think>`.)

**Fix for a re-run:** raise `max_length` (Qwen3 and Llama-3.1 accept far longer contexts), or set
`tokenizer.truncation_side = "left"` so only the oldest context is dropped.

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import time
import gc
from tqdm import tqdm


# 4-bit NormalFloat quantization (≈5–6 GB of VRAM per 8B model).
def get_bnb_config() -> BitsAndBytesConfig:
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )


# Load tokenizer + 4-bit model; left padding for batched generation.
def load_model_4bit(model_name: str, model_id: str):
    print(f"\n{'='*60}")
    print(f"⏳ Loading {model_name}")
    print(f"   HF ID : {model_id}")
    print(f"{'='*60}")

    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, token=os.environ.get("HF_TOKEN"), trust_remote_code=True,
    )
    tokenizer.padding_side = "left"  # decoder-only models must be left-padded in a batch
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quant_config = get_bnb_config() if USE_4BIT else None
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map="auto",
        token=os.environ.get("HF_TOKEN"),
        trust_remote_code=True,
        attn_implementation="flash_attention_2",  # faster attention kernels
    )
    model.eval()

    load_time = time.time() - t0
    vram_used = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f"✅ {model_name} ready | Load: {load_time:.1f}s | VRAM: {vram_used:.1f} GB")
    return tokenizer, model


# System prompt (JSON only) + the sample's input as the user turn.
def build_prompt(sample: dict, model_name: str, tokenizer) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert Agile project manager. "
                "When asked to output JSON, respond ONLY with valid JSON — "
                "no markdown fences, no explanations, no preamble."
            ),
        },
        {"role": "user", "content": sample["input"]},
    ]
    try:
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
    except Exception:
        formatted = (
            "System: You are an expert Agile project manager.\n\n"
            f"User: {sample['input']}\n\nAssistant:"
        )
    if "Qwen3" in model_name or "qwen3" in model_name.lower():
        # Qwen3 soft switch to skip <think> (here it lands on the role line; outputs still show an empty <think></think>)
        formatted = formatted.replace("<|im_start|>user", "<|im_start|>user /no_think", 1)
    return formatted


# Generate answers for the whole dataset in batches.
def run_batched_inference(tokenizer, model, dataset: list, model_name: str, batch_size: int = BATCH_SIZE) -> list:
    results = []
    n_batches = (len(dataset) + batch_size - 1) // batch_size
    print(f"\n🚀 Running batched inference: {len(dataset)} samples, batch_size={batch_size}")

    prompts = [build_prompt(s, model_name, tokenizer) for s in dataset]
    total_tokens = 0
    wall_t0 = time.time()

    for batch_idx in tqdm(range(n_batches), desc=f"{model_name}", ncols=80):
        start = batch_idx * batch_size
        end   = min(start + batch_size, len(dataset))
        batch_prompts = prompts[start:end]
        batch_samples = dataset[start:end]

        # ⚠️ prompts > 1,024 tokens lose their END (see note above)
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[1]

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
                do_sample=DO_SAMPLE, pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.05,
            )
        batch_latency = time.time() - t0

        for sample, output_ids in zip(batch_samples, outputs):
            generated_ids = output_ids[input_len:]  # only the new tokens
            response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            n_tokens = int((generated_ids != tokenizer.eos_token_id).sum())  # pad = eos, so padding is not counted
            total_tokens += n_tokens

            results.append({
                "sample_id": sample["id"], "task_type": sample["task_type"],
                "input": sample["input"], "reference": sample["reference_output"],
                "response": response,
                "latency_s": round(batch_latency / len(batch_samples), 2),  # batch time ÷ batch size
                "tokens_gen": n_tokens,
                "tokens_per_sec": round(n_tokens / (batch_latency / len(batch_samples)), 1),
                # carry through real-track metadata for scoring
                # real ground-truth fields for Cell 5
                "_meta": {k: v for k, v in sample.items() if k not in ("id", "task_type", "input", "reference_output")},
            })

    wall_elapsed = time.time() - wall_t0
    print(f"\n✅ {model_name} inference done | {wall_elapsed/60:.1f} min | {total_tokens:,} tokens")
    return results


# Free the GPU before the next model is loaded.
def unload_model(model, tokenizer):
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"🗑️  Model unloaded | VRAM now: {torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0:.1f} GB")


print("✅ Local inference engine ready (unchanged from v5)")

✅ Local inference engine ready (unchanged from v5)


## Cell 4b — Gemini API Inference

Sequential calls through the new `google-genai` client with the same system prompt, temperature and token limit
as the local models. Each call is retried up to 3 times (2 s, 4 s back-off) and followed by a 5.45 s pause.
The result dict has the **same shape** as the local models' results, so the scoring and judge code is shared.

* `tokens_gen` / `tokens_per_sec` are `None` — the SDK's token counts aren't comparable with local generation.
* `latency_s` is a network round-trip, not GPU time.
* Gemini's prompt is sent as plain `contents` — **no 1,024-token truncation**, which is one reason its
  task-allocation results differ so much from the local models.

In [ ]:
import os
import time as _time
from google import genai
from google.genai import types
from tqdm import tqdm

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])  # one client for the whole run

GEMINI_SYSTEM_PROMPT = (
    "You are an expert Agile project manager. "
    "When asked to output JSON, respond ONLY with valid JSON — "
    "no markdown fences, no explanations, no preamble."
)


# Sequential API inference with retries; same result schema as the local models.
def run_gemini_inference(dataset: list, model_id: str, model_name: str,
                          max_retries: int = 3, sleep_between: float = None) -> list:
    """
    Sequential API inference (no local batching — API has its own
    concurrency handling). Retries on transient errors, matches the same
    result schema as run_batched_inference so it merges cleanly.
    """
    sleep_between = GEMINI_SLEEP_BETWEEN if sleep_between is None else sleep_between

    # Define generation parameters using the new structured configuration schema
    # Same system prompt and sampling settings as the local models.
    config = types.GenerateContentConfig(
        system_instruction=GEMINI_SYSTEM_PROMPT,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_NEW_TOKENS,
    )

    results = []
    print(f"\n🚀 Running Gemini inference: {len(dataset)} samples "
          f"| rate limit: {GEMINI_REQUESTS_PER_MIN} req/min ({sleep_between:.2f}s between calls)")
    for sample in tqdm(dataset, desc=model_name, ncols=80):
        response_text, latency = "", 0.0
        for attempt in range(max_retries):
            t0 = _time.time()
            try:
                # Use client.models.generate_content for the new SDK structure
                resp = client.models.generate_content(
                    model=model_id,
                    contents=sample["input"],  # full prompt, not truncated
                    config=config
                )
                response_text = (resp.text or "").strip()
                latency = _time.time() - t0
                break
            except Exception as e:
                latency = _time.time() - t0
                if attempt == max_retries - 1:
                    print(f"\n⚠️  Gemini failed on {sample['id']}: {e}")
                    response_text = ""
                else:
                    _time.sleep(2 * (attempt + 1))  # back-off 2 s, then 4 s

        results.append({
            "sample_id": sample["id"], "task_type": sample["task_type"],
            "input": sample["input"], "reference": sample["reference_output"],
            "response": response_text,
            "latency_s": round(latency, 2),
            "tokens_gen": None,       # not exposed uniformly by the API
            "tokens_per_sec": None,
            "_meta": {k: v for k, v in sample.items() if k not in ("id", "task_type", "input", "reference_output")},
        })
        _time.sleep(sleep_between)  # be polite to the API rate limit  # rate limit: 11 requests / minute

    print(f"\n✅ Gemini inference done")
    return results


print("✅ Gemini API inference ready")


✅ Gemini API inference ready


## Cell 5 — Real-Track Metrics Engine

**Objective:** score each answer against what really happened, track by track.

| Track | Function | What is compared |
|---|---|---|
| all | `check_json_validity` | answer parses as JSON · share of required keys present |
| all | `detect_hallucination` | refusal / apology phrases |
| task design | `score_task_design` | predicted `priority` (capitalised) == real Jira priority |
| task allocation | `score_task_allocation` | predicted `assigned_to` == real assignee (case-insensitive) · `|pred−real| / real` hours |
| sprint planning | `score_sprint_planning` | fuzzy title match (`difflib` ratio ≥ 0.6) between predicted and real issue lists → precision / recall / F1 · scope MAPE |
| resource planning | `score_resource_planning` | **first number** in `capacity_constraints` vs real velocity |

`compute_real_metrics` parses the reference JSON (for sprint / resource ground truth), runs the shared checks and
dispatches to the track scorer.

### ⚠️ Reviewer notes on the metrics

* **`assignee_plausible_match` is identical to the exact match** — the roster/skill lookup was never wired in.
  Report only the exact match.
* **Velocity MAPE is not informative.** The prompt *states* the real velocity ("observed real velocity of 81.3
  issues per release"), so a model that repeats it scores 0 error. Llama and Gemini get MAPE ≈ 0.000 for exactly
  this reason; Qwen's 0.40 comes from a different first number in its text.
* **Hours MAPE is dominated by outliers** — real logged hours are often tiny, so a sensible 16 h estimate against
  0.5 h logged gives MAPE = 31. Report the median, not the mean.
* `_fuzzy_match` with threshold 0.6 can match two different issues that share many words.

In [ ]:
import re, difflib
from rouge_score import rouge_scorer  # kept for optional diagnostic use, not scored into composites  # imported for optional diagnostics only


# Strip fences, try a full parse, else the greedy {…} / […] block (replaced by a better version in Cell 9).
def extract_json(text: str):
    clean = re.sub(r"```(?:json)?\s*", "", text).replace("```", "").strip()
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        pass
    match = re.search(r"(\{.*\}|\[.*\])", clean, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    return None


# Valid JSON? Which required keys are present / missing?
def check_json_validity(response: str, task_type: str) -> dict:
    parsed = extract_json(response)
    if parsed is None:
        return {"valid_json": False, "schema_compliance": 0.0,
                "missing_fields": REQUIRED_FIELDS.get(task_type, [])}
    required = REQUIRED_FIELDS.get(task_type, [])
    check_obj = parsed[0] if isinstance(parsed, list) and parsed else parsed  # a list answer → check its first element
    if not isinstance(check_obj, dict):
        return {"valid_json": True, "schema_compliance": 0.0, "missing_fields": required}
    present = [f for f in required if f in check_obj]
    missing = [f for f in required if f not in check_obj]
    return {"valid_json": True,
            "schema_compliance": round(len(present) / len(required), 3) if required else 1.0,
            "missing_fields": missing}


# Refusal / apology phrases counted as 'hallucination'.
HALLUC_PATTERNS = [
    r"I (don't|do not|cannot|can't) (know|provide|access)",
    r"As an AI", r"I apologize", r"I'm sorry",
]

def detect_hallucination(response: str) -> bool:
    return any(re.search(p, response, re.IGNORECASE) for p in HALLUC_PATTERNS)


# Similarity ratio of two strings (0–1); ≥ 0.6 counts as the same issue.
def _fuzzy_match(a: str, b: str, threshold: float = 0.6) -> bool:
    return difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio() >= threshold


# First number in a free-text string (used for velocity and '40 hours').
def _first_number(text: str):
    m = re.search(r"[-+]?\d*\.?\d+", text or "")
    return float(m.group(0)) if m else None


# Absolute percentage error |pred − real| / |real|; None if either side is missing or real = 0.
def _mape(pred, real):
    if pred is None or real in (None, 0):
        return None
    return round(abs(pred - real) / abs(real), 4)


# ── Track-specific scorers ────────────────────────────────────────────────

# Track 1: does the predicted priority equal the real Jira priority?
def score_task_design(parsed: dict, meta: dict) -> dict:
    pred_priority = str((parsed or {}).get("priority", "")).strip().capitalize()  # 'HIGH' / 'high' → 'High' to match Jira's spelling
    known = meta.get("known_priority")
    return {
        "priority_predicted": pred_priority,
        "priority_known": known,
        "priority_match": bool(known) and pred_priority == known,
    }


# Track 2: right person? how far off are the hours?
def score_task_allocation(parsed: dict, meta: dict) -> dict:
    pred_assignee = str((parsed or {}).get("assigned_to", "")).strip()
    pred_hours    = (parsed or {}).get("estimated_hours")
    try:
        pred_hours = float(pred_hours) if pred_hours is not None else None
    except (TypeError, ValueError):
        pred_hours = None

    real_assignee = meta.get("ground_truth_assignee")
    real_hours    = meta.get("ground_truth_hours")

    exact_match = bool(real_assignee) and pred_assignee.strip().lower() == real_assignee.strip().lower()
    # plausible match: best-effort — same as exact unless a roster/skill lookup
    # is wired in; exact match is the primary metric, this is reported alongside.
    plausible_match = exact_match  # placeholder — no roster/skill check implemented

    return {
        "assignee_predicted": pred_assignee,
        "assignee_real": real_assignee,
        "assignee_exact_match": exact_match,
        "assignee_plausible_match": plausible_match,
        "hours_predicted": pred_hours,
        "hours_real": real_hours,
        "hours_mape": _mape(pred_hours, real_hours),
    }


# Track 3: overlap between predicted and real release contents.
def score_sprint_planning(parsed: dict, meta: dict) -> dict:
    pred_tasks = (parsed or {}).get("selected_tasks", [])
    if not isinstance(pred_tasks, list):
        pred_tasks = []
    pred_tasks = [str(t) for t in pred_tasks]

    real_ref = meta.get("reference_output_parsed") or {}
    real_tasks = real_ref.get("selected_tasks", [])
    real_scope = real_ref.get("scope_size")

    matched = 0
    used = set()
    # Greedy one-to-one matching: each real issue can be matched once.
    for pt in pred_tasks:
        for i, rt in enumerate(real_tasks):
            if i in used:
                continue
            if _fuzzy_match(pt, rt):
                matched += 1
                used.add(i)
                break

    precision = matched / len(pred_tasks) if pred_tasks else 0.0
    recall    = matched / len(real_tasks) if real_tasks else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0  # harmonic mean of precision and recall

    pred_scope = (parsed or {}).get("scope_size")
    try:
        pred_scope = float(pred_scope) if pred_scope is not None else None
    except (TypeError, ValueError):
        pred_scope = None

    return {
        "taskset_precision": round(precision, 3),
        "taskset_recall": round(recall, 3),
        "taskset_f1": round(f1, 3),
        "scope_predicted": pred_scope,
        "scope_real": real_scope,
        "scope_mape": _mape(pred_scope, real_scope),
    }


# Track 4: velocity read from free text (low confidence, see note).
def score_resource_planning(parsed: dict, meta: dict) -> dict:
    capacity_text = str((parsed or {}).get("capacity_constraints", ""))
    pred_velocity = _first_number(capacity_text)  # best-effort, low-confidence

    real_ref = meta.get("reference_output_parsed") or {}
    real_velocity = real_ref.get("real_avg_velocity")

    return {
        "velocity_predicted_bestguess": pred_velocity,
        "velocity_real": real_velocity,
        "velocity_mape": _mape(pred_velocity, real_velocity),
        "velocity_confidence": "low — extracted from free text, not a dedicated field",
    }


# task_type → scorer function.
TRACK_SCORERS = {
    "task_design": score_task_design,
    "task_allocation": score_task_allocation,
    "sprint_planning": score_sprint_planning,
    "resource_planning": score_resource_planning,
}


# Shared checks + the track-specific scorer for one result.
def compute_real_metrics(result: dict) -> dict:
    response  = result["response"]
    task_type = result["task_type"]
    meta      = dict(result.get("_meta", {}))

    # pre-parse the real reference_output JSON string, if present, for the
    # track scorers that need structured real fields (sprint/resource)
    ref_raw = result.get("reference", "")
    try:
        meta["reference_output_parsed"] = json.loads(ref_raw) if ref_raw else {}  # real ground truth for sprint / resource tracks
    except json.JSONDecodeError:
        meta["reference_output_parsed"] = {}

    json_check = check_json_validity(response, task_type)
    parsed     = extract_json(response)
    halluc     = detect_hallucination(response)

    track_metrics = TRACK_SCORERS[task_type](parsed if isinstance(parsed, dict) else {}, meta)  # a non-dict answer is scored as empty

    return {
        "auto_valid_json":        json_check["valid_json"],
        "auto_schema_compliance": json_check["schema_compliance"],
        "auto_missing_fields":    json_check["missing_fields"],
        "auto_hallucination":     halluc,
        **track_metrics,
    }


print("✅ Real-track metrics engine ready")

✅ Real-track metrics engine ready


## Cell 6 — LLM Judge (adapted for real samples)

Same parallel judge as the synthetic benchmark (one thread per API key, mid-run checkpoints), with two changes:

* The prompt includes the **input** and tells the judge when there is **no reference** (task design), so it grades
  plausibility instead of similarity.
* A strict JSON-only system prompt and a 4-step fallback parser (`_parse_judge_response`) make judge replies easier
  to read.

| Truncation applied before judging | Characters |
|---|---|
| input prompt | 600 |
| reference | 500 |
| **model answer** | **500** |

`_sanitize` also replaces `{ }` with `( )` so the text cannot break `.format()`.

> ⚠️ With only the first 500 characters of an answer visible, "completeness" is judged on a fragment for most
> sprint and resource plans. A failed judge call is stored as verdict `"ERROR"` with empty scores (155 of 800 here).

In [ ]:
import requests
import copy
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# One worker thread per key.
INCEPTION_API_KEYS = [
    "YOUR_INCEPTION_API_KEY_1",
    "YOUR_INCEPTION_API_KEY_2",
    "YOUR_INCEPTION_API_KEY_3",
]

REQUESTS_PER_MIN = 60  # per-key throttle
SLEEP_BETWEEN_CALLS = 60.0 / REQUESTS_PER_MIN

# Forces a bare JSON reply.
JUDGE_SYSTEM = (
    "You are a strict JSON-only evaluator. "
    "You MUST respond with a single valid JSON object and absolutely nothing else. "
    "No markdown fences, no preamble, no explanation, no trailing text. "
    "Only the raw JSON object starting with { and ending with }."
)

# Rubric: 5 dimensions + overall (1–5). {{ }} are literal braces.
JUDGE_PROMPT_TEMPLATE = """\
Evaluate this AI-generated project management output.

TASK TYPE: {task_type}

INPUT:
{input_prompt}

{reference_section}

MODEL OUTPUT:
{model_output}

Score 1-5 on each dimension (1=very poor, 5=excellent).
Return ONLY this JSON object with integer scores filled in:
{{"relevance": 0, "completeness": 0, "feasibility": 0, "clarity": 0, "agile_alignment": 0, "overall_score": 0.0, "strengths": "", "weaknesses": "", "verdict": ""}}
"""

# What is stored when every attempt fails.
_JUDGE_NULL = {
    "relevance": None, "completeness": None, "feasibility": None,
    "clarity": None, "agile_alignment": None, "overall_score": None,
    "strengths": "JUDGE_ERROR", "weaknesses": "JUDGE_ERROR", "verdict": "ERROR",
}


# Truncate and neutralise braces.
def _sanitize(text: str, limit: int) -> str:
    return (text or "")[:limit].replace("{", "(").replace("}", ")")


# Try 4 increasingly forgiving ways to read the judge's JSON.
def _parse_judge_response(content: str) -> dict:
    clean = re.sub(r"```(?:json)?\s*", "", content).replace("```", "").strip()
    for fn in [
        lambda s: json.loads(s),
        lambda s: json.loads(re.search(r"\{[^{}]*\}", s, re.DOTALL).group(0)),
        lambda s: json.loads(s[s.find("{"):s.rfind("}")+1]),
        lambda s: json.loads(re.sub(r",\s*([}\]])", r"\1", s[s.find("{"):s.rfind("}")+1])),  # last resort: drop trailing commas
    ]:
        try:
            return fn(clean)
        except Exception:
            pass
    return None


# Build the prompt for one sample and call the judge (3 attempts).
def _judge_one(result: dict, api_key: str, retries: int = 3) -> dict:
    ref = result.get("reference", "")
    if ref.strip():
        reference_section = f"REFERENCE OUTPUT (partial real ground truth):\n{_sanitize(ref, 500)}"  # sprint / resource / allocation
    else:
        reference_section = (
            "REFERENCE OUTPUT: none available — this is a real-world sample with no "
            "authored reference. Grade plausibility and quality of the output given "
            "the INPUT alone."
        )

    prompt = JUDGE_PROMPT_TEMPLATE.format(
        task_type=result["task_type"],
        input_prompt=_sanitize(result["input"], 600),
        reference_section=reference_section,
        model_output=_sanitize(result["response"], 500),  # ⚠️ judge sees only the first 500 chars of the answer
    )
    payload = {
        "model": JUDGE_MODEL,
        "messages": [
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.0,
        "max_tokens": 512,
    }

    for attempt in range(retries):
        try:
            resp = requests.post(
                INCEPTION_URL,
                headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
                json=payload, timeout=30,
            )
            resp.raise_for_status()
            parsed = _parse_judge_response(resp.json()["choices"][0]["message"]["content"].strip())
            if parsed:
                return parsed
        except requests.exceptions.HTTPError:
            if resp.status_code == 429:  # rate-limited → exponential back-off 5 s, 10 s, 20 s
                backoff = 5 * (2 ** attempt)
                print(f"\n⚠️  429 on key ...{api_key[-6:]} — backing off {backoff}s")
                time.sleep(backoff)
            else:
                print(f"\n⚠️  HTTP {resp.status_code}")
        except Exception:
            time.sleep(1 * (attempt + 1))

    return copy.copy(_JUDGE_NULL)


# One thread: judge its partition, copy scores into the result dicts.
def _worker(partition: list, api_key: str, pbar, lock, errors_counter: list):
    for result in partition:
        j = _judge_one(result, api_key)
        result["judge_relevance"]       = j.get("relevance")
        result["judge_completeness"]    = j.get("completeness")
        result["judge_feasibility"]     = j.get("feasibility")
        result["judge_clarity"]         = j.get("clarity")
        result["judge_agile_alignment"] = j.get("agile_alignment")
        result["judge_overall"]         = j.get("overall_score")
        result["judge_strengths"]       = j.get("strengths")
        result["judge_weaknesses"]      = j.get("weaknesses")
        result["judge_verdict"]         = j.get("verdict", "ERROR")
        if result["judge_verdict"] == "ERROR":
            with lock:
                errors_counter[0] += 1
        with lock:
            pbar.update(1)
        time.sleep(SLEEP_BETWEEN_CALLS)


# Split un-judged results into contiguous chunks, one per key; checkpoint every 150.
def run_judge_phase(all_results: list, save_every: int = 150) -> list:
    needs_judge = [r for r in all_results if r.get("judge_verdict") is None]
    n_keys = len(INCEPTION_API_KEYS)
    if not needs_judge:
        print("✅ All samples already judged.")
        return all_results

    chunk = (len(needs_judge) + n_keys - 1) // n_keys  # contiguous chunks (Notebook 4 uses round-robin)
    parts = [needs_judge[i*chunk:(i+1)*chunk] for i in range(n_keys)]
    parts = [p for p in parts if p]

    print(f"\n🧑‍⚖️  Parallel judge | {len(needs_judge)} samples | {len(parts)} keys")

    lock = threading.Lock()
    errors_counter = [0]

    with tqdm(total=len(needs_judge), desc="Judge", ncols=80) as pbar:
        with ThreadPoolExecutor(max_workers=len(parts)) as executor:
            futures = [executor.submit(_worker, part, key, pbar, lock, errors_counter)
                       for part, key in zip(parts, INCEPTION_API_KEYS)]
            saved_at = 0
            while any(not f.done() for f in futures):
                done_now = sum(1 for r in needs_judge if r.get("judge_verdict") is not None)
                if done_now - saved_at >= save_every:
                    with open(os.path.join(CHECKPOINT_DIR, f"judge_midrun_{done_now}.json"), "w") as f:
                        json.dump(all_results, f)  # mid-run safety checkpoint
                    saved_at = done_now
                time.sleep(5)
            for f in futures:
                f.result()

    print(f"\n✅ Judge complete | errors: {errors_counter[0]} / {len(needs_judge)}")
    return all_results


print(f"✅ Judge ready | {len(INCEPTION_API_KEYS)} keys | {REQUESTS_PER_MIN} req/min/key")

✅ Judge ready | 3 keys | 60 req/min/key


## Cell 7 — Benchmark Runner (local models + Gemini)

For each local model: load from checkpoint if present, otherwise 4-bit load → batched inference →
`compute_real_metrics` → save checkpoint → unload. Gemini follows the same pattern without load/unload. Every
result row starts with empty judge fields, which Cell 8's judge phase fills in.

In [ ]:
def run_inference_phase(dataset: list, local_models: dict, api_models: dict) -> tuple:
    run_id = "real_001"  # suffix for checkpoint and CSV names
    all_results = []

    # ── Local (HF, 4-bit) models ──────────────────────────────────────────
    for model_name, model_id in local_models.items():
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_{run_id}.json")
        # Resume: skip models that already have a checkpoint.
        if os.path.exists(ckpt_path):
            with open(ckpt_path) as f:
                model_results = json.load(f)
            print(f"\n♻️  {model_name}: loaded {len(model_results)} results from checkpoint")
            all_results.extend(model_results)
            continue

        tokenizer, model = load_model_4bit(model_name, model_id)
        raw_results = run_batched_inference(tokenizer, model, dataset, model_name)

        model_results = []
        for raw in raw_results:
            metrics = compute_real_metrics(raw)  # real-track scoring (Cell 5)
            model_results.append({
                "model": model_name, "sample_id": raw["sample_id"], "task_type": raw["task_type"],
                "input": raw["input"], "reference": raw["reference"], "response": raw["response"],
                "latency_s": raw["latency_s"], "tokens_gen": raw["tokens_gen"], "tokens_per_sec": raw["tokens_per_sec"],
                **metrics,
                "judge_relevance": None, "judge_completeness": None, "judge_feasibility": None,
                "judge_clarity": None, "judge_agile_alignment": None, "judge_overall": None,
                "judge_strengths": None, "judge_weaknesses": None, "judge_verdict": None,
            })

        with open(ckpt_path, "w") as f:
            json.dump(model_results, f)
        print(f"  💾 Checkpoint saved: {ckpt_path}")
        all_results.extend(model_results)
        unload_model(model, tokenizer)  # free VRAM before the next model

    # ── API (Gemini) models ────────────────────────────────────────────────
    for model_name, model_id in api_models.items():
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_{run_id}.json")
        if os.path.exists(ckpt_path):
            with open(ckpt_path) as f:
                model_results = json.load(f)
            print(f"\n♻️  {model_name}: loaded {len(model_results)} results from checkpoint")
            all_results.extend(model_results)
            continue

        raw_results = run_gemini_inference(dataset, model_id, model_name)  # no GPU work for the API model

        model_results = []
        for raw in raw_results:
            metrics = compute_real_metrics(raw)
            model_results.append({
                "model": model_name, "sample_id": raw["sample_id"], "task_type": raw["task_type"],
                "input": raw["input"], "reference": raw["reference"], "response": raw["response"],
                "latency_s": raw["latency_s"], "tokens_gen": raw["tokens_gen"], "tokens_per_sec": raw["tokens_per_sec"],
                **metrics,
                "judge_relevance": None, "judge_completeness": None, "judge_feasibility": None,
                "judge_clarity": None, "judge_agile_alignment": None, "judge_overall": None,
                "judge_strengths": None, "judge_weaknesses": None, "judge_verdict": None,
            })

        with open(ckpt_path, "w") as f:
            json.dump(model_results, f)
        print(f"  💾 Checkpoint saved: {ckpt_path}")
        all_results.extend(model_results)

    return all_results, run_id


print("✅ Benchmark runner ready (local + API dispatch)")

✅ Benchmark runner ready (local + API dispatch)


## Cell 8 — Execute the Full Real-Data Benchmark

In [ ]:
print("🚀 Starting REAL-data benchmark...")
print(f"   Local models : {list(MODELS.keys())}")
print(f"   API models   : {list(API_MODELS.keys())}")
print(f"   Samples      : {len(REAL_BENCHMARK_DATASET)} (50 per task type)")
print()

wall_start = time.time()

# Phase 1 — inference (local models + Gemini)
# Phase 1 — 4 models × 200 samples
all_results, run_id = run_inference_phase(REAL_BENCHMARK_DATASET, MODELS, API_MODELS)

# Phase 2 — judge
all_results = run_judge_phase(all_results)  # Phase 2 — judge all 800 answers

# Save full results
df_results = pd.DataFrame(all_results)
output_csv = f"real_benchmark_results_{run_id}.csv"  # the file every later cell reads
df_results.to_csv(output_csv, index=False)

elapsed = (time.time() - wall_start) / 3600
print(f"\n✅ Real-data benchmark complete!")
print(f"   Wall time : {elapsed:.2f} hours")
print(f"   Results   : {output_csv} ({len(df_results)} rows)")

🚀 Starting REAL-data benchmark...
   Local models : ['Qwen3-8B', 'GLM-Z1-9B', 'Llama-3.1-8B']
   API models   : ['Gemini-3.5-Flash-Lite']
   Samples      : 200 (50 per task type)


♻️  Qwen3-8B: loaded 200 results from checkpoint

♻️  GLM-Z1-9B: loaded 200 results from checkpoint

⏳ Loading Llama-3.1-8B
   HF ID : meta-llama/Llama-3.1-8B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (10 outputs left out)

Gemini-3.5-Flash-Lite: 100%|██████████████████| 200/200 [23:12<00:00,  6.96s/it]



✅ Gemini inference done
  💾 Checkpoint saved: checkpoints_real/Gemini-3.5-Flash-Lite_real_001.json

🧑‍⚖️  Parallel judge | 800 samples | 3 keys


Judge: 100%|██████████████████████████████████| 800/800 [19:10<00:00,  1.44s/it]


✅ Judge complete | errors: 155 / 800

✅ Real-data benchmark complete!
   Wall time : 0.88 hours
   Results   : real_benchmark_results_real_001.csv (800 rows)


### 📊 Reading the output

* Qwen3-8B and GLM-Z1-9B were loaded from checkpoints of an earlier run; Llama-3.1-8B (9.2 min, 63,556 tokens)
  and Gemini (200 calls) ran in this session.
* `✅ Judge complete | errors: 155 / 800` — about one judge score in five is missing; the per-track tables below
  average only the judged rows.
* Wall time **0.88 h** for the whole run.

## Cell 9 — Fix `extract_json`, re-score, inspect, per-track tables

This section has four cells.

**9a — the fix.** The original `extract_json` used a greedy regex from the *first* `{` to the *last* `}`. When an
answer contains several JSON objects (e.g. the model restates the team roster, then gives its answer), that span is
not valid JSON and the real answer is lost. The new version tries `json.JSONDecoder.raw_decode` at **every** `{` / `[`
and keeps the **last** dict found. `score_task_allocation` also learns to read `"40 hours"` as 40.

**9b — re-score saved answers** with the fixed functions (no new inference, no new judging) → `real_benchmark_results_fixed.csv`.

**9c — look at raw answers** to understand why task-allocation scores are so low.

**9d — per-track tables.**

In [ ]:
# ── FIX: extract_json no longer over-captures across multiple JSON objects ──
#
# Root cause: the old greedy regex r"(\{.*\}|\[.*\])" with re.DOTALL matches
# from the FIRST '{' to the LAST '}' in the whole response. Task Allocation's
# prompt embeds a multi-object team roster JSON, and models frequently
# restate/reference it before their final answer — so the response contains
# MULTIPLE JSON objects, and the old regex spanned all of them into one
# unparseable blob, silently discarding a perfectly valid final answer.
#
# Fix: scan every '{'/'[' position, attempt a real JSON decode from each,
# and keep the LAST successfully-parsed dict (the final answer almost always
# comes after any restated context, not before).

import json as _json  # local alias in case `json` was shadowed elsewhere in the notebook

# Robust version: try every '{' / '[' start position, keep the LAST dict.
def extract_json(text: str):
    clean = re.sub(r"```(?:json)?\s*", "", text).replace("```", "").strip()

    # Fast path: already a single clean JSON blob (most well-behaved responses)
    try:
        return _json.loads(clean)
    except _json.JSONDecodeError:
        pass

    # Robust path: scan for every candidate JSON value in the text, keep the
    # LAST dict found (handles restated context / multiple JSON blocks).
    decoder = _json.JSONDecoder()  # raw_decode parses ONE JSON value starting at a given index
    candidates = []
    for i, ch in enumerate(clean):
        if ch in "{[":
            try:
                obj, _end = decoder.raw_decode(clean, i)
                candidates.append(obj)
            except _json.JSONDecodeError:
                continue

    if candidates:
        dict_candidates = [c for c in candidates if isinstance(c, dict)]
        return dict_candidates[-1] if dict_candidates else candidates[-1]  # the final answer usually comes last

    return None


# ── Hardening: estimated_hours parsing now tolerates non-bare-number strings ──
# Secondary issue worth fixing alongside: if a model writes "estimated_hours":
# "40 hours" (string, not a bare number), float("40 hours") raises and the old
# code just gave up (pred_hours = None). Now falls back to extracting the
# first number from the string before giving up entirely.

# Same scorer as Cell 5, but '40 hours' → 40.0 instead of None.
def score_task_allocation(parsed: dict, meta: dict) -> dict:
    pred_assignee = str((parsed or {}).get("assigned_to", "")).strip()
    raw_hours     = (parsed or {}).get("estimated_hours")

    pred_hours = None
    if raw_hours is not None:
        try:
            pred_hours = float(raw_hours)
        except (TypeError, ValueError):
            pred_hours = _first_number(str(raw_hours))  # e.g. "40 hours" -> 40.0  # fallback for strings like '40 hours'

    real_assignee = meta.get("ground_truth_assignee")
    real_hours    = meta.get("ground_truth_hours")

    exact_match = bool(real_assignee) and pred_assignee.strip().lower() == real_assignee.strip().lower()
    plausible_match = exact_match  # unchanged — see earlier note on roster/skill lookup

    return {
        "assignee_predicted": pred_assignee,
        "assignee_real": real_assignee,
        "assignee_exact_match": exact_match,
        "assignee_plausible_match": plausible_match,
        "hours_predicted": pred_hours,
        "hours_real": real_hours,
        "hours_mape": _mape(pred_hours, real_hours),
    }

# re-register in case TRACK_SCORERS dict was already built with the old function
TRACK_SCORERS["task_allocation"] = score_task_allocation  # make compute_real_metrics use the new scorer

print("✅ extract_json + score_task_allocation patched")

✅ extract_json + score_task_allocation patched


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Recompute real-track metrics on ALREADY-SAVED responses (no re-inference,
# no re-judging needed — reuses the "response"/"reference" text you already
# have on disk, just re-runs the FIXED scoring functions over it).
#
# Run this AFTER the patch above (fix_extract_json_bug.py) in the same session.
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np

# Load your existing combined results (adjust path if yours differs)
df_results = pd.read_csv(output_csv) if "output_csv" in dir() else pd.read_csv("real_benchmark_results_real_001.csv")  # the original (un-fixed) results

print(f"Recomputing metrics for {len(df_results)} rows using the fixed extract_json...")


# CSV turns empty strings into NaN; turn them back into ''.
def _safe_str(val) -> str:
    """CSV round-trips turn empty strings into NaN (a float). json.loads(NaN)
    crashes with a TypeError, and NaN is truthy in plain Python so `if val:`
    doesn't catch it either — coerce explicitly instead."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return ""
    return str(val)


# NaN → None for ground-truth fields.
def _safe_meta_value(val):
    """Same NaN issue for meta fields (known_priority, ground_truth_*) — but
    here we want None (missing), not empty string, since downstream code
    checks `bool(known)` / `meta.get(...)` and treats None as 'absent'."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    return val


recomputed_rows = []
for _, row in df_results.iterrows():
    result = {
        "response": _safe_str(row.get("response")),
        "task_type": row["task_type"],
        "reference": _safe_str(row.get("reference")),
        "_meta": {
            # ⚠️ the CSV column is 'priority_known' → None → priority_match all False in the fixed CSV
            "known_priority": _safe_meta_value(row.get("known_priority")),
            "ground_truth_assignee": _safe_meta_value(
                row.get("assignee_real") if "assignee_real" in row else row.get("ground_truth_assignee")
            ),
            "ground_truth_hours": _safe_meta_value(
                row.get("hours_real") if "hours_real" in row else row.get("ground_truth_hours")
            ),
        },
    }
    new_metrics = compute_real_metrics(result)  # re-score with the fixed functions
    recomputed_rows.append(new_metrics)

recomputed_df = pd.DataFrame(recomputed_rows)

# Overwrite only the metric columns, keep everything else (model, sample_id,
# input, response, judge scores, etc.) exactly as they were.
for col in recomputed_df.columns:
    df_results[col] = recomputed_df[col].values

df_results.to_csv("real_benchmark_results_fixed.csv", index=False)  # written, but NOT re-loaded by the cells below

# Quick before/after sanity check specifically for the bug we just fixed
print("\n" + "="*70)
print("hours_mape coverage AFTER fix (should be > 0 now for local SLMs)")
print("="*70)
ta = df_results[df_results["task_type"] == "task_allocation"]
coverage = ta.groupby("model")["hours_mape"].apply(lambda s: s.notna().sum())
total_n  = ta.groupby("model").size()
print(pd.DataFrame({"valid_hours_mape_n": coverage, "total_n": total_n}))

print(f"\n💾 Saved corrected results -> real_benchmark_results_fixed.csv")
print("Re-run Cell 9 / 10 / 11 / 12 pointing at this file to get corrected tables/charts.")

Recomputing metrics for 800 rows using the fixed extract_json...

hours_mape coverage AFTER fix (should be > 0 now for local SLMs)
                       valid_hours_mape_n  total_n
model                                             
GLM-Z1-9B                               0       50
Gemini-3.5-Flash-Lite                  50       50
Llama-3.1-8B                            0       50
Qwen3-8B                                0       50

💾 Saved corrected results -> real_benchmark_results_fixed.csv
Re-run Cell 9 / 10 / 11 / 12 pointing at this file to get corrected tables/charts.


### 📊 Reading the output

`valid_hours_mape_n` is still **0 / 50** for all three local models after the fix, and 50 / 50 for Gemini. So the
JSON-extraction bug was not the reason — the next cell shows the real cause.

### ⚠️ Reviewer notes

* The fixed scores are saved to `real_benchmark_results_fixed.csv`, but Cells 9c, 9d, 10, 11 and 12 all re-load
  `output_csv` (the **original** file). The tables printed below therefore use the original `extract_json`.
* If you do switch to the fixed CSV, first correct the `known_priority` lookup (the CSV column is `priority_known`),
  otherwise priority match becomes 0 for every model. Notebook 7 does the re-scoring with this corrected.

In [ ]:
import pandas as pd

df_results = pd.read_csv(output_csv) if "output_csv" in dir() else pd.read_csv("real_benchmark_results_real_001.csv")

ta = df_results[df_results["task_type"] == "task_allocation"]

# Print the first 2 task-allocation answers of each local model.
for model in ["Qwen3-8B", "Llama-3.1-8B", "GLM-Z1-9B"]:
    print("="*70)
    print(model)
    print("="*70)
    sample_rows = ta[ta["model"] == model].head(2)
    for _, row in sample_rows.iterrows():
        print(f"--- sample_id: {row['sample_id']} ---")
        print(repr(row["response"])[:1500])  # repr() shows literal \n, quotes etc — important for diagnosis  # repr() shows \n and quotes literally
        print()

Qwen3-8B
--- sample_id: real_alloc_CASSANDRA-21574 ---
'role": "Test/Unit Engineer",\n    "skills": [\n      "Test/unit",\n      "Test/dtest/java",\n      "Test/dtest/python",\n      "Local/Other"\n    ],\n    "capacity": 8\n  }\n]\n\nPlease assign the task to the most appropriate person based on their skills and capacity.\n</think>\n\n{\n  "task": "Fix failing tests for TrackerReconcilerTest",\n  "required_component": "Accord",\n  "complexity": "medium",\n  "assigned_to": "David Capwell",\n  "reasoning": "David Capwell is the only team member with explicit \'Accord\' skill listed in their skills, which matches the required component. He also has CI and Build skills which are relevant for test fixes, and his capacity is available."\n}'

--- sample_id: real_alloc_HADOOP-19453 ---
'"role": "fs/s3 Engineer",\n    "skills": [\n      "fs/s3",\n      "test"\n    ],\n    "capacity": 8\n  },\n  {\n    "name": "Rohit Goyal",\n    "role": "fs/s3 Engineer",\n    "skills": [\n      "fs/s3",\n     

### 📊 Reading the output — the root cause

Qwen's answer starts with `'role": "Test/Unit Engineer", "skills": [...`, Llama's and GLM's with similar fragments:
the models are **continuing the team-roster JSON from the prompt**. That is exactly what happens when the
end of the prompt (instruction + assistant header) has been cut off by `max_length=1024` (see the Cell 4 note).
Some answers then contain a `<think>` block and an answer later on, but most never reach a usable JSON answer.

**Conclusion for the paper:** the local models' task-allocation scores (assignee match 0–2 %, judge ≈ 1.1–1.3) are a
**pipeline artefact**, not evidence that 8B models cannot allocate tasks. Re-run task allocation with a longer
`max_length` before drawing conclusions on that track.

In [ ]:
df_results = pd.read_csv(output_csv) if "output_csv" in dir() else pd.read_csv("real_benchmark_results_real_001.csv")  # original (un-fixed) CSV — see note above

print("="*70)
print("TASK DESIGN — Schema Compliance · Priority-Match · Judge")
print("="*70)
td = df_results[df_results["task_type"] == "task_design"]  # Track 1
print(td.groupby("model")[["auto_schema_compliance", "priority_match", "judge_overall"]].mean().round(3))

print("\n" + "="*70)
print("TASK ALLOCATION — Assignee Match · Hours MAPE · Judge")
print("="*70)
ta = df_results[df_results["task_type"] == "task_allocation"]  # Track 2
print(ta.groupby("model")[["assignee_exact_match", "hours_mape", "judge_overall"]].mean().round(3))

print("\n" + "="*70)
print("SPRINT PLANNING — Task-Set F1 · Scope-Size MAPE · Judge")
print("="*70)
sp = df_results[df_results["task_type"] == "sprint_planning"]  # Track 3
print(sp.groupby("model")[["taskset_precision", "taskset_recall", "taskset_f1", "scope_mape", "judge_overall"]].mean().round(3))

print("\n" + "="*70)
print("RESOURCE PLANNING — Velocity MAPE (low-confidence) · Schema · Judge")
print("="*70)
rp = df_results[df_results["task_type"] == "resource_planning"]  # Track 4
print(rp.groupby("model")[["velocity_mape", "auto_schema_compliance", "judge_overall"]].mean().round(3))

TASK DESIGN — Schema Compliance · Priority-Match · Judge
                       auto_schema_compliance priority_match  judge_overall
model                                                                      
GLM-Z1-9B                                0.24           0.08          1.746
Gemini-3.5-Flash-Lite                    1.00           0.48          2.823
Llama-3.1-8B                             0.82            0.4          2.690
Qwen3-8B                                 0.98           0.48          2.128

TASK ALLOCATION — Assignee Match · Hours MAPE · Judge
                      assignee_exact_match  hours_mape  judge_overall
model                                                                
GLM-Z1-9B                             0.02         NaN          1.138
Gemini-3.5-Flash-Lite                 0.24      30.732          2.156
Llama-3.1-8B                           0.0         NaN          1.139
Qwen3-8B                               0.0         NaN          1.260

SPRINT PLAN

### 📊 Reading the output

| Track | Gemini-3.5-Flash-Lite | Llama-3.1-8B | Qwen3-8B | GLM-Z1-9B |
|---|---|---|---|---|
| Task design — schema / priority / judge | 1.00 / 0.48 / **2.82** | 0.82 / 0.40 / 2.69 | 0.98 / 0.48 / 2.13 | 0.24 / 0.08 / 1.75 |
| Task allocation — exact assignee / judge | **0.24** / **2.16** | 0.00 / 1.14 | 0.00 / 1.26 | 0.02 / 1.14 |
| Sprint planning — task-set F1 / scope MAPE / judge | 0.06 / 0.64 / **2.74** | **0.28** / **0.37** / 2.46 | 0.27 / 0.39 / 2.41 | 0.00 / – / 1.40 |
| Resource planning — schema / judge | 1.00 / 1.72 | 0.36 / **1.73** | 1.00 / 1.55 | 0.06 / 1.25 |

* **Task design:** Qwen and Gemini predict the real priority equally often (48 %). But 27 of the 50 sampled issues
  are really `High`, so a trivial "always answer High" rule would score **54 %** — better than every model. Qwen
  answers `High` 43 times out of 50. Priority match should be reported against this majority-class baseline.
* **Sprint planning:** Llama and Qwen recover ~30 % of the issues that really shipped. Gemini's sprint plans are
  long: **46 of 50 stop mid-JSON at the 600-token limit**, so few parse and its F1 drops to 0.06, even though the judge
  (which sees only the first 500 characters) likes them. `MAX_NEW_TOKENS = 600` is too small for real sprint plans.
* **Resource planning:** every model is judged poorly (≤ 1.73). Only Gemini and Qwen return all three required keys.
* GLM-Z1-9B is last everywhere: it usually spends its 600-token budget on reasoning and never finishes the JSON.

## Cell 10 — Statistical Tests per Track

* **Wilcoxon signed-rank** (paired by `sample_id`) on each track's primary metric: judge for task design and resource
  planning, exact assignee match for task allocation, task-set F1 for sprint planning. Values are coerced to numbers
  and pairs with all-zero differences are skipped (Wilcoxon is undefined there).
* **McNemar's test** for the binary assignee match: only the *discordant* pairs (one model right, the other wrong)
  matter. The exact binomial version is used when there are fewer than 25 discordant pairs.

> Requires `statsmodels` (`from statsmodels.stats.contingency_tables import mcnemar`).

In [ ]:
from scipy import stats
from itertools import combinations
import numpy as np
import pandas as pd

# Paired Wilcoxon for every model pair on one metric (robust to NaN / object dtypes).
def wilcoxon_pairwise(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    models = df["model"].unique()
    rows = []
    for m1, m2 in combinations(models, 2):
        pivot = (df[df["model"].isin([m1, m2])]
                 .pivot_table(index="sample_id", columns="model", values=metric, dropna=False))

        if m1 not in pivot.columns or m2 not in pivot.columns:
            print(f"⚠️  {m1} or {m2} has NO valid '{metric}' values at all — skipping")
            continue

        pivot = pivot[[m1, m2]].dropna()
        pivot[m1] = pd.to_numeric(pivot[m1], errors="coerce").astype(np.float64)  # booleans / strings from the CSV → floats
        pivot[m2] = pd.to_numeric(pivot[m2], errors="coerce").astype(np.float64)
        pivot = pivot.dropna()

        if len(pivot) < 10:
            print(f"⚠️  Too few paired samples for {m1} vs {m2} on {metric} (n={len(pivot)})")
            continue

        diffs = pivot[m1] - pivot[m2]
        if (diffs == 0).all():  # Wilcoxon cannot run when every difference is zero
            print(f"⚠️  {m1} vs {m2} on {metric}: all paired differences are zero — skipping")
            continue

        stat, p = stats.wilcoxon(pivot[m1], pivot[m2])
        diff = pivot[m1].mean() - pivot[m2].mean()
        rows.append({
            "Model A": m1, "Model B": m2,
            "Mean A": round(pivot[m1].mean(), 4), "Mean B": round(pivot[m2].mean(), 4),
            "Δ (A–B)": round(diff, 4), "p-value": round(p, 4),
            "Significant": "✅ Yes" if p < 0.05 else "❌ No",
            "Better": m1 if diff > 0 else m2,
        })
    return pd.DataFrame(rows)


# Which metric is tested on each track.
TRACK_PRIMARY_METRIC = {
    "task_design": "judge_overall",
    "task_allocation": "assignee_exact_match",
    "sprint_planning": "taskset_f1",
    "resource_planning": "judge_overall",
}

for task_type, metric in TRACK_PRIMARY_METRIC.items():
    sub = df_results[df_results["task_type"] == task_type]
    print("="*70)
    print(f"{task_type.upper()} — metric: {metric}")
    print("="*70)
    wdf = wilcoxon_pairwise(sub, metric)
    if not wdf.empty:
        print(wdf.to_string(index=False))
    print()

print("="*70)
print("DIAGNOSTIC — hours_mape coverage (task_allocation), for reference only")
print("="*70)
ta = df_results[df_results["task_type"] == "task_allocation"]
coverage = ta.groupby("model")["hours_mape"].apply(lambda s: s.notna().sum())
total_n  = ta.groupby("model").size()
print(pd.DataFrame({"valid_hours_mape_n": coverage, "total_n": total_n}))

# ── McNemar's test for task_allocation's assignee_exact_match ──────────────
# FIX: previous version named each row's discordant-count columns after the
# specific model pair (e.g. "Qwen3-8B right / GLM-Z1-9B wrong"). Since every
# row in the loop has a DIFFERENT pair of models, pd.DataFrame(rows) unioned
# all those differently-named columns together, leaving each row's OTHER
# pairs' columns as NaN — a wall of NaN, not a bug in the underlying test.
# Fix: use generic, pair-agnostic column names ("A_right_B_wrong" /
# "B_right_A_wrong") so every row populates the SAME columns.
from statsmodels.stats.contingency_tables import mcnemar

# McNemar's test for a paired 0/1 metric.
def mcnemar_pairwise(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    models = df["model"].unique()
    rows = []
    for m1, m2 in combinations(models, 2):
        pivot = (df[df["model"].isin([m1, m2])]
                 .pivot_table(index="sample_id", columns="model", values=metric, dropna=False))
        if m1 not in pivot.columns or m2 not in pivot.columns:
            print(f"⚠️  {m1} or {m2} has NO valid '{metric}' values — skipping McNemar")
            continue
        pivot = pivot[[m1, m2]].dropna()
        pivot[m1] = pd.to_numeric(pivot[m1], errors="coerce").astype(np.float64)
        pivot[m2] = pd.to_numeric(pivot[m2], errors="coerce").astype(np.float64)
        pivot = pivot.dropna()

        if len(pivot) < 10:
            print(f"⚠️  Too few paired samples for {m1} vs {m2} on {metric} (n={len(pivot)})")
            continue

        both_right   = int(((pivot[m1] == 1) & (pivot[m2] == 1)).sum())
        a_right_only = int(((pivot[m1] == 1) & (pivot[m2] == 0)).sum())  # discordant: only A correct
        b_right_only = int(((pivot[m1] == 0) & (pivot[m2] == 1)).sum())  # discordant: only B correct
        both_wrong   = int(((pivot[m1] == 0) & (pivot[m2] == 0)).sum())

        discordant = a_right_only + b_right_only
        if discordant == 0:
            print(f"⚠️  {m1} vs {m2} on {metric}: zero discordant pairs — no difference to test")
            continue

        table = [[both_right, a_right_only], [b_right_only, both_wrong]]
        use_exact = discordant < 25  # exact binomial test for small counts
        result = mcnemar(table, exact=use_exact, correction=not use_exact)

        rows.append({
            "Model A": m1, "Model B": m2,
            "A_right_B_wrong": a_right_only,   # generic, pair-agnostic column names —
            "B_right_A_wrong": b_right_only,   # same two columns for every row, no more union-NaN
            "p-value": round(result.pvalue, 4),
            "Significant": "✅ Yes" if result.pvalue < 0.05 else "❌ No",
            "Better": m1 if a_right_only > b_right_only else m2,
        })
    return pd.DataFrame(rows)


print("="*70)
print("TASK_ALLOCATION — McNemar's test (assignee_exact_match, paired binary)")
print("  A_right_B_wrong = # samples where Model A got the real assignee right and B didn't")
print("  B_right_A_wrong = # samples where Model B got it right and A didn't")
print("="*70)
ta = df_results[df_results["task_type"] == "task_allocation"]
mcdf = mcnemar_pairwise(ta, "assignee_exact_match")
if not mcdf.empty:
    print(mcdf.to_string(index=False))
print()

TASK_DESIGN — metric: judge_overall
     Model A               Model B  Mean A  Mean B  Δ (A–B)  p-value Significant                Better
    Qwen3-8B             GLM-Z1-9B  2.1000  1.7162   0.3838   0.0062       ✅ Yes              Qwen3-8B
    Qwen3-8B          Llama-3.1-8B  2.1160  2.6080  -0.4920   0.0005       ✅ Yes          Llama-3.1-8B
    Qwen3-8B Gemini-3.5-Flash-Lite  2.2042  2.7958  -0.5917   0.0017       ✅ Yes Gemini-3.5-Flash-Lite
   GLM-Z1-9B          Llama-3.1-8B  1.7071  2.7000  -0.9929   0.0000       ✅ Yes          Llama-3.1-8B
   GLM-Z1-9B Gemini-3.5-Flash-Lite  1.7607  2.8107  -1.0500   0.0000       ✅ Yes Gemini-3.5-Flash-Lite
Llama-3.1-8B Gemini-3.5-Flash-Lite  2.6526  2.7105  -0.0579   0.7575        ❌ No Gemini-3.5-Flash-Lite

TASK_ALLOCATION — metric: assignee_exact_match
⚠️  Qwen3-8B vs Llama-3.1-8B on assignee_exact_match: all paired differences are zero — skipping
     Model A               Model B  Mean A  Mean B  Δ (A–B)  p-value Significant                Be

### 📊 Reading the output

* **Task design (judge):** Llama-3.1-8B ≈ Gemini (p = 0.76, not significant); both beat Qwen3-8B and GLM-Z1-9B
  (p ≤ 0.002). Qwen beats GLM (p = 0.006).
* **Task allocation (assignee match, McNemar):** Gemini is right on 11–12 samples where each local model is wrong,
  and never the other way round (p ≤ 0.001). Between the local models there is nothing to test (all ≈ 0) — see the
  truncation note.
* **Sprint planning (task-set F1):** Qwen ≈ Llama (p = 0.73); both beat Gemini and GLM (p < 0.001).
* **Resource planning (judge):** Llama ≈ Gemini ≈ Qwen (all p > 0.07); only GLM is significantly worse.
* Notebook 7 repeats the tests with the fine-tuned model included.

## Cell 11 — Export Paper Tables (one per track)

`export_track_table` averages the chosen columns per model, saves `paper_table_<track>.csv` and prints a LaTeX
table with caption and label. Same numbers as Cell 9d.

In [ ]:
# One CSV + one LaTeX table per track.
def export_track_table(task_type: str, cols: list, name: str):
    sub = df_results[df_results["task_type"] == task_type]
    summary = sub.groupby("model")[cols].mean().round(3)
    print(f"\n{name}")
    print(summary.to_string())
    summary.to_csv(f"paper_table_{task_type}.csv")  # e.g. paper_table_task_design.csv
    print(summary.to_latex(float_format="%.3f", caption=name, label=f"tab:{task_type}"))
    return summary


export_track_table("task_design", ["auto_schema_compliance", "priority_match", "judge_overall"],
                    "Real-World Task Design (N per model = 50, reference-free)")

export_track_table("task_allocation", ["assignee_exact_match", "hours_mape", "judge_overall"],
                    "Real-World Task Allocation (N per model = 50, real assignee/hours ground truth)")

export_track_table("sprint_planning", ["taskset_f1", "scope_mape", "judge_overall"],
                    "Real-World Sprint Planning (N per model = 50, fixVersion-as-sprint)")

export_track_table("resource_planning", ["velocity_mape", "auto_schema_compliance", "judge_overall"],
                    "Real-World Resource Planning (N per model = 50, velocity = issues/release)")


Real-World Task Design (N per model = 50, reference-free)
                       auto_schema_compliance priority_match  judge_overall
model                                                                      
GLM-Z1-9B                                0.24           0.08          1.746
Gemini-3.5-Flash-Lite                    1.00           0.48          2.823
Llama-3.1-8B                             0.82            0.4          2.690
Qwen3-8B                                 0.98           0.48          2.128
\begin{table}
\caption{Real-World Task Design (N per model = 50, reference-free)}
\label{tab:task_design}
\begin{tabular}{lrlr}
\toprule
 & auto_schema_compliance & priority_match & judge_overall \\
model &  &  &  \\
\midrule
GLM-Z1-9B & 0.240 & 0.080 & 1.746 \\
Gemini-3.5-Flash-Lite & 1.000 & 0.480 & 2.823 \\
Llama-3.1-8B & 0.820 & 0.400 & 2.690 \\
Qwen3-8B & 0.980 & 0.480 & 2.128 \\
\bottomrule
\end{tabular}
\end{table}


Real-World Task Allocation (N per model = 50, real assign

,velocity_mape,auto_schema_compliance,judge_overall
model,,,
GLM-Z1-9B,0.001,0.06,1.251
Gemini-3.5-Flash-Lite,0.000,1.00,1.721
Llama-3.1-8B,0.000,0.36,1.730
Qwen3-8B,0.400,1.00,1.550


## Cell 12 — Visualizations (per-track bars + judge radar)

Four grouped bar charts (one per track, each on its own scale) and a judge radar across all models.

> ⚠️ The four bar charts are saved, but `plot_judge_radar` stops with
> `TypeError: agg function failed [how->mean, dtype->object]`: after the CSV round-trip some judge columns contain
> text, so `.mean()` fails. Converting them first fixes it:
> `df[avail] = df[avail].apply(pd.to_numeric, errors="coerce")`. Notebook 7 draws the radar with that fix.
>
> Also note that `hours_mape` (values ≈ 30) shares an axis with 0–1 metrics in the task-allocation chart, which
> flattens the other bars.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Keys match the model names in the data (no gray fallback here).
MODEL_COLORS = {
    "Qwen3-8B":              "#1E88E5",
    "Llama-3.1-8B":          "#43A047",
    "GLM-Z1-9B":             "#E64A19",
    "Gemini-3.5-Flash-Lite": "#8E24AA",
}
BG   = "#FFFFFF"   # white background
FG   = "#1a1a1a"   # dark text/ticks/titles, readable on white
GRID = "#dddddd"   # light gray gridlines/spines


def fig_setup(w=10, h=6):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.tick_params(colors=FG)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["bottom", "left"]].set_color(GRID)
    return fig, ax


# Grouped bars for one track: one group per metric, one bar per model.
def plot_track_bars(df: pd.DataFrame, task_type: str, metrics: list, labels: list,
                     title: str, filename: str, higher_is_better: list = None):
    """Grouped bar chart for one track's metrics, one bar-group per model."""
    sub = df[df["task_type"] == task_type]
    agg = sub.groupby("model")[metrics].mean()
    models = [m for m in MODEL_COLORS if m in agg.index]  # keep consistent order

    fig, ax = fig_setup(10, 5.5)
    n_metrics = len(metrics)
    n_models  = len(models)
    bar_w     = 0.8 / n_models
    x         = np.arange(n_metrics)

    for i, model in enumerate(models):
        vals = agg.loc[model, metrics].values
        offset = (i - (n_models - 1) / 2) * bar_w  # centre the bars of each group on its tick
        ax.bar(x + offset, vals, width=bar_w, label=model,
               color=MODEL_COLORS.get(model, "gray"))

    ax.set_xticks(x)
    ax.set_xticklabels(labels, color=FG, fontsize=10)
    ax.set_title(title, color=FG, fontsize=13, pad=15)
    ax.legend(loc="upper right", labelcolor=FG, facecolor="white", edgecolor=GRID, fontsize=8)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, facecolor=BG)
    plt.show()
    print(f"💾 saved {filename}")


# Radar of the 5 judge dimensions — comparable across tracks.
def plot_judge_radar(df: pd.DataFrame):
    """Judge-score radar — the one metric computed identically across all four tracks."""
    dims   = ["judge_relevance", "judge_completeness", "judge_feasibility",
              "judge_clarity", "judge_agile_alignment"]
    labels = ["Relevance", "Completeness", "Feasibility", "Clarity", "Agile\nAlign"]
    avail  = [d for d in dims if d in df.columns]
    if not avail:
        print("No judge scores available yet.")
        return

    agg = df.groupby("model")[avail].mean()  # ⚠️ fails here when judge columns are object dtype
    N   = len(avail)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.grid(color=GRID, linewidth=0.5)
    ax.spines["polar"].set_color(GRID)

    for model in agg.index:
        vals = agg.loc[model].tolist() + [agg.loc[model, avail[0]]]
        c = MODEL_COLORS.get(model, "gray")
        ax.plot(angles, vals, "o-", lw=2, color=c, label=model)
        ax.fill(angles, vals, alpha=0.12, color=c)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, color=FG, fontsize=10)
    ax.set_ylim(0, 5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(["1", "2", "3", "4", "5"], color="#666666", fontsize=8)
    ax.set_title("LLM-as-Judge Scores Across All 4 Models (1–5)", color=FG, fontsize=13, pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1),
              labelcolor=FG, facecolor="white", edgecolor=GRID)
    plt.tight_layout()
    plt.savefig("radar_judge_real.png", dpi=150, facecolor=BG)
    plt.show()
    print("💾 saved radar_judge_real.png")


# ── Generate all charts ────────────────────────────────────────────────────

plot_track_bars(
    df_results, "task_design",
    metrics=["auto_schema_compliance", "priority_match", "judge_overall"],
    labels=["Schema\nCompliance", "Priority\nMatch", "Judge/5"],
    title="Real-World Task Design", filename="chart_task_design.png",
)

plot_track_bars(
    df_results, "task_allocation",
    metrics=["assignee_exact_match", "hours_mape", "judge_overall"],
    labels=["Assignee\nExact Match", "Hours\nMAPE (lower=better)", "Judge/5"],
    title="Real-World Task Allocation", filename="chart_task_allocation.png",
)

plot_track_bars(
    df_results, "sprint_planning",
    metrics=["taskset_f1", "scope_mape", "judge_overall"],
    labels=["Task-Set F1", "Scope MAPE\n(lower=better)", "Judge/5"],
    title="Real-World Sprint Planning", filename="chart_sprint_planning.png",
)

plot_track_bars(
    df_results, "resource_planning",
    metrics=["velocity_mape", "auto_schema_compliance", "judge_overall"],
    labels=["Velocity MAPE\n(low-confidence)", "Schema\nCompliance", "Judge/5"],
    title="Real-World Resource Planning", filename="chart_resource_planning.png",
)

plot_judge_radar(df_results)  # raises the TypeError shown below

💾 saved chart_task_design.png
💾 saved chart_task_allocation.png
💾 saved chart_sprint_planning.png
💾 saved chart_resource_planning.png


TypeError: agg function failed [how->mean,dtype->object]